# 第 5 章 · 会话、状态与记忆：ADK 的三层记忆体系

> 一个能投入使用的 Agent 必须"记得"。ADK 把"记忆"拆成三个层次，各自解决不同时间尺度的问题：
>
> | 层次 | 时间尺度 | 类比 | ADK 构件 |
> |---|---|---|---|
> | **Session（会话）** | 一次对话 | 一次就诊的完整病历 | `SessionService` |
> | **State（状态）** | 会话内/跨会话的结构化数据 | 病历上的关键指标栏 | `session.state` |
> | **Memory（长期记忆）** | 跨会话的知识沉淀 | 病人的历史健康档案 | `MemoryService` |

```mermaid
flowchart TB
    subgraph RUN["一次 Agent 运行"]
        EV["Event 事件流<br/>（每条消息/每次工具调用）"]
        SD["state_delta 状态增量"]
    end
    subgraph SS["SessionService 会话服务"]
        S1["Session: app+user+session_id<br/>├── events: 对话历史<br/>└── state: 键值状态"]
    end
    subgraph MS["MemoryService 记忆服务"]
        M1["跨会话可检索的记忆库"]
    end
    RUN --> SS
    SS -->|"add_session_to_memory"| MS
    MS -.->|"search_memory 召回"| RUN
    style S1 fill:#e8f0fe,stroke:#4285f4,stroke-width:2px
    style M1 fill:#e6f4ea,stroke:#34a853,stroke-width:2px
```

---

## 1. Session：一次对话的完整档案

`Session` = `app_name` + `user_id` + `session_id` 三元组定位的一份档案，内含：

- `events`：从第一句"你好"开始的所有事件（对话历史）；
- `state`：伴随会话演化的键值状态；
- `last_update_time` 等元数据。

`SessionService` 是可插拔的：

| 实现 | 用途 |
|---|---|
| `InMemorySessionService` | 学习与原型（进程退出即丢失） |
| `DatabaseSessionService` | 生产：SQLAlchemy 支持的各种数据库 |
| `VertexAiSessionService` | 托管：Vertex AI Agent Engine |

> 💡 前几章我们已经在用 Session 了——同一个 `session_id` 下的多轮对话就是它在工作。本章聚焦**另外两个层次**。


In [1]:
import os
assert os.environ.get("DEEPSEEK_API_KEY"), "请先设置 DEEPSEEK_API_KEY"

from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import ToolContext
from google.genai import types

APP, USER = "adk_ch05", "student"
session_service = InMemorySessionService()

async def run(root_agent, query, session_id, initial_state=None, show_state=True):
    try:
        await session_service.create_session(app_name=APP, user_id=USER,
                                             session_id=session_id, state=initial_state or {})
    except Exception:
        pass
    runner = Runner(agent=root_agent, app_name=APP, session_service=session_service)
    msg = types.Content(role="user", parts=[types.Part(text=query)])
    print(f"🧑 {query}")
    async for ev in runner.run_async(user_id=USER, session_id=session_id, new_message=msg):
        if ev.is_final_response() and ev.content and ev.content.parts:
            print(f"🤖 {ev.content.parts[0].text}")
    session = await session_service.get_session(app_name=APP, user_id=USER, session_id=session_id)
    if show_state:
        print(f"🗂️ 当前 state → {dict(session.state)}\n" + "─" * 50)
    return session

print("✅ 环境就绪")


✅ 环境就绪


---

## 2. State：带"作用域前缀"的工作记忆

`state` 是会话中的键值对，但 ADK 给它设计了一个精巧的**作用域系统**——键名的前缀决定数据的生存范围：

| 前缀 | 作用域 | 生存范围 | 典型用途 |
|---|---|---|---|
| （无前缀） | Session | 仅当前会话 | 本次任务的中间产物 |
| `user:` | 用户级 | **跨会话**，跟随 user_id | 用户偏好、画像 |
| `app:` | 应用级 | 全局共享 | 全局配置 |
| `temp:` | 临时 | 仅当前这一轮运行，不落盘 | 中间计算 |

状态的写入方式我们已经见过两种：`create_session(state=...)` 初始化、`tool_context.state[...] = ...` 工具内写入。每次写入都会生成一个 `state_delta` 附加在事件的 `actions` 上，随事件流广播——这就是第 4 章多智能体能"接力"的底层机制。

### 实战：用 `user:` 前缀实现跨会话记忆

下面这个 Agent 会在工具里把用户偏好写入 `user:` 作用域；之后我们**换一个全新会话**，验证它依然生效：


In [2]:
def remember_preference(item: str, value: str, tool_context: ToolContext) -> dict:
    """记录用户的一项偏好（跨会话长期保存）。

    Args:
        item: 偏好项目，例如 "饮品"、"音乐"。
        value: 偏好内容，例如 "乌龙茶"。
    """
    tool_context.state[f"user:pref_{item}"] = value   # user: 前缀 → 跨会话
    return {"status": f"已记住你对「{item}」的偏好"}

def recall_preferences(tool_context: ToolContext) -> dict:
    """查看当前已记住的所有用户偏好。"""
    prefs = {k: v for k, v in tool_context.state.to_dict().items() if k.startswith("user:pref_")}
    return {"preferences": prefs or "（还没有记录）"}

butler = Agent(
    name="butler", model=LiteLlm(model="deepseek/deepseek-chat"),
    instruction="你是贴身管家：用户提到喜好就调用 remember_preference；问偏好就调用 recall_preferences；回答时优先参考已知偏好。",
    description="跨会话记住用户偏好的管家",
    tools=[remember_preference, recall_preferences],
)

# 会话 A：告诉管家偏好
await run(butler, "记住：我最喜欢的饮品是乌龙茶。", session_id="session_A")


🧑 记住：我最喜欢的饮品是乌龙茶。


D:\Python\Lib\site-packages\google\adk\models\llm_request.py:273: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  declaration = tool._get_declaration()


22:59:33 - LiteLLM:WARNING: get_model_cost_map.py:289 - LiteLLM: Failed to fetch remote model cost map from https://raw.githubusercontent.com/BerriAI/litellm/main/model_prices_and_context_window.json: The read operation timed out. Falling back to local backup.


🤖 好的，我已经记住了！您最喜欢的饮品是**乌龙茶** ☕，以后我会记住这个偏好的。还有什么需要我帮忙的吗？
🗂️ 当前 state → {'user:pref_饮品': '乌龙茶'}
──────────────────────────────────────────────────


Session(id='session_A', app_name='adk_ch05', user_id='student', state={'user:pref_饮品': '乌龙茶'}, events=[Event(model_version=None, content=Content(
  parts=[
    Part(
      text='记住：我最喜欢的饮品是乌龙茶。'
    ),
  ],
  role='user'
), grounding_metadata=None, partial=None, turn_complete=None, turn_complete_reason=None, finish_reason=None, error_code=None, error_message=None, interrupted=None, custom_metadata=None, usage_metadata=None, live_session_resumption_update=None, live_session_id=None, go_away=None, voice_activity=None, input_transcription=None, output_transcription=None, avg_logprobs=None, logprobs_result=None, cache_metadata=None, citation_metadata=None, interaction_id=None, environment_id=None, invocation_id='e-e640005f-13df-41f8-a87f-6c75452a22a5', author='user', actions=EventActions(skip_summarization=None, state_delta={}, artifact_delta={}, transfer_to_agent=None, escalate=None, requested_auth_configs={}, requested_tool_confirmations={}, compaction=None, end_of_agent=None, agent_stat

In [3]:
# 会话 B：全新 session_id，但 user: 前缀的数据应该还在
await run(butler, "我们以前聊过吗？你知道我喜欢喝什么吗？", session_id="session_B")


🧑 我们以前聊过吗？你知道我喜欢喝什么吗？


🤖 当然记得！我知道你喜欢喝**乌龙茶**。 

这是我们之前聊过时记下的偏好。如果你还有其他喜好想让我记住，随时告诉我哦！😊
🗂️ 当前 state → {'user:pref_饮品': '乌龙茶'}
──────────────────────────────────────────────────


Session(id='session_B', app_name='adk_ch05', user_id='student', state={'user:pref_饮品': '乌龙茶'}, events=[Event(model_version=None, content=Content(
  parts=[
    Part(
      text='我们以前聊过吗？你知道我喜欢喝什么吗？'
    ),
  ],
  role='user'
), grounding_metadata=None, partial=None, turn_complete=None, turn_complete_reason=None, finish_reason=None, error_code=None, error_message=None, interrupted=None, custom_metadata=None, usage_metadata=None, live_session_resumption_update=None, live_session_id=None, go_away=None, voice_activity=None, input_transcription=None, output_transcription=None, avg_logprobs=None, logprobs_result=None, cache_metadata=None, citation_metadata=None, interaction_id=None, environment_id=None, invocation_id='e-652ce46b-3433-48fc-939c-69011d417e4b', author='user', actions=EventActions(skip_summarization=None, state_delta={}, artifact_delta={}, transfer_to_agent=None, escalate=None, requested_auth_configs={}, requested_tool_confirmations={}, compaction=None, end_of_agent=None, agent_

> 🔍 注意看第二个单元格打印的 state：`user:pref_饮品` 出现在了**全新会话**里。框架在创建会话时自动把 `user:` / `app:` 作用域的数据合并进来——这就是 ADK 对"跨会话状态"的原生支持，**无需数据库代码**（内存服务范围内）。

---

## 3. Memory：可检索的长期记忆

State 是**结构化**的记忆（精确的键值），Memory 则是**非结构化**的记忆（可语义/关键词检索的历史沉淀）。典型流程：

```mermaid
flowchart LR
    S["会话结束<br/>（或阶段性结束）"] --> ADD["memory_service<br/>.add_session_to_memory(session)"]
    ADD --> DB[("记忆库")]
    DB --> SEARCH["后续会话中<br/>search_memory(query)"]
    SEARCH --> CTX["注入上下文<br/>Agent 引经据典"]
    style DB fill:#e6f4ea,stroke:#34a853,stroke-width:2px
```

`MemoryService` 的实现：

| 实现 | 检索方式 | 用途 |
|---|---|---|
| `InMemoryMemoryService` | 关键词匹配 | 学习原型 |
| `VertexAiMemoryBankService` | 语义检索（托管） | 生产 |
| `VertexAiRagMemoryService` | RAG 语料库 | 接入自有知识库 |

下面演示完整闭环：会话 1 沉淀记忆 → 会话 2 中 Agent 通过 `search_memory` 工具召回：


In [4]:
from google.adk.memory import InMemoryMemoryService

memory_service = InMemoryMemoryService()

# 第一步：在"归档会话"里聊一些事实
await run(butler, "补充一下：我对花生过敏，请一定记住。", session_id="archive_session")

# 第二步：把会话沉淀进记忆库
archive = await session_service.get_session(app_name=APP, user_id=USER, session_id="archive_session")
await memory_service.add_session_to_memory(archive)
print("📥 会话已沉淀到记忆库\n" + "─" * 50)

# 第三步：直接检索验证（框架内部也是走这个 API）
results = await memory_service.search_memory(app_name=APP, user_id=USER, query="过敏")
for m in results.memories:
    for p in m.content.parts:
        if p.text:
            print("🔍 命中记忆：", p.text[:80])


🧑 补充一下：我对花生过敏，请一定记住。


🤖 已为您记住，这是一条非常重要的信息：

- **过敏：花生过敏**

我会特别留意这一点，在未来的任何推荐、用餐安排或相关建议中都会避开花生及含花生成分的物品，确保您的安全。请放心！如果您还有其他偏好，欢迎随时告诉我。
🗂️ 当前 state → {'user:pref_饮品': '乌龙茶', 'user:pref_过敏': '花生过敏'}
──────────────────────────────────────────────────
📥 会话已沉淀到记忆库
──────────────────────────────────────────────────
🔍 命中记忆： 已为您记住，这是一条非常重要的信息：

- **过敏：花生过敏**

我会特别留意这一点，在未来的任何推荐、用餐安排或相关建议中都会避开花生及含花生成分的物品，


In [5]:
# 第四步：给 Agent 装备记忆检索能力——PreloadMemoryTool 会在每轮自动召回相关记忆
from google.adk.tools.preload_memory_tool import PreloadMemoryTool

wise_butler = Agent(
    name="wise_butler", model=LiteLlm(model="deepseek/deepseek-chat"),
    instruction="你是贴身管家。回答任何问题前，注意检查自动召回的记忆（若有），并严格遵守其中记录的用户禁忌与偏好。",
    description="有长期记忆的管家",
    tools=[PreloadMemoryTool()],
)

runner = Runner(agent=wise_butler, app_name=APP,
                session_service=session_service, memory_service=memory_service)  # ← 注入 memory_service
await session_service.create_session(app_name=APP, user_id=USER, session_id="fresh_session")
msg = types.Content(role="user", parts=[types.Part(text="给我推荐一款下午茶点心。")])
async for ev in runner.run_async(user_id=USER, session_id="fresh_session", new_message=msg):
    if ev.is_final_response() and ev.content:
        print("🤖", ev.content.parts[0].text)


🤖 好的，下午茶确实需要一点精致感来搭配。我建议先从适合搭配茶或咖啡的甜点入手。


> 🔍 如果 Agent 在推荐时避开了花生（或主动提及过敏禁忌），说明 `PreloadMemoryTool` 成功召回了上一个会话沉淀的记忆。**注意**：`InMemoryMemoryService` 是关键词匹配，召回能力有限；生产环境的语义检索请换 `VertexAiMemoryBankService` 或自建 RAG 方案。

---

## 4. Artifact：管理"文件级"内容

State 适合存**小数据**（字符串、数字、小 JSON）；当 Agent 需要处理**文件**（图片、PDF、生成的大文档）时，应该用 `ArtifactService`：

```python
# 在工具中保存一个文件（示意代码）
async def save_report(text: str, tool_context: ToolContext) -> dict:
    artifact = types.Part.from_text(text=text)
    version = await tool_context.save_artifact(filename="report.md", artifact=artifact)
    return {"saved": "report.md", "version": version}
```

| 实现 | 用途 |
|---|---|
| `InMemoryArtifactService` | 学习原型 |
| `GcsArtifactService` | 生产：Google Cloud Storage |

Artifact 带**版本管理**（同名文件多次保存自动累加版本号），适合"报告反复修订"类场景。

---

## 5. 三层体系速查与选型

| 我要存什么？ | 用哪层？ | 键/接口 |
|---|---|---|
| 本次任务的中间产物 | Session State（无前缀） | `state["draft"]` |
| 用户偏好（跨会话） | State `user:` 前缀 | `state["user:theme"]` |
| 全局配置 | State `app:` 前缀 | `state["app:version"]` |
| 历史对话的可检索沉淀 | Memory | `add_session_to_memory` / `search_memory` |
| 文件与大型二进制 | Artifact | `save_artifact` / `load_artifact` |

---

## 6. 与 LangGraph 对照 🔄

| ADK | LangGraph | 差异点评 |
|---|---|---|
| Session（events + state） | Thread（Checkpointer 持久化的 State） | 高度对应：都是"对话单元 + 持久化" |
| `state` + 作用域前缀 | State 的各字段（作用域靠节点逻辑自己实现） | ADK 的前缀是贴心语法糖 |
| `MemoryService` | Store（跨线程长期记忆，langgraph.store） | 概念一致；LangGraph Store 支持命名空间+语义检索 |
| `ArtifactService` | 无直接对应（存 State 或外部存储） | ADK 特有的小亮点 |
| `PreloadMemoryTool` | 节点里调用 store.search() | ADK 自动化程度更高 |

---

## 📌 本章要点回顾

- 三层记忆：**Session**（对话档案）→ **State**（结构化键值，含 `user:`/`app:`/`temp:` 作用域）→ **Memory**（可检索长期沉淀）；
- `user:` 前缀一行代码实现跨会话偏好；
- 记忆闭环：会话 → `add_session_to_memory` → `search_memory` / `PreloadMemoryTool` 召回；
- 大文件走 Artifact，小数据走 State，历史沉淀走 Memory。

> ➡️ 下一章：[06-回调-观测与部署](06-回调-观测与部署.ipynb) —— 把 Agent 打磨成生产级系统。


---

## 🧪 本章练习

### 1. 状态作用域实验（基础）

分别写入普通键、`user:` 键、`app:` 键和临时状态（如当前版本支持），建立两个用户、每个用户两个 Session，逐项验证哪些值会共享、哪些值会隔离。将结果整理成“写入位置—可见范围—适用数据—不适用数据”表格。

### 2. 跨会话用户偏好（进阶）

构建餐厅推荐助手：在一个 Session 中记录饮食偏好和过敏原，在新 Session 中仍能使用这些信息；另一用户不能读取该偏好。加入“忘记我的偏好”请求，并说明删除、覆盖或过期策略应由哪一层实现。

### 3. 完成记忆闭环并评估召回（进阶）

准备五段含相似关键词但语义不同的历史对话，依次完成 Session 沉淀到 Memory、查询或预加载、基于召回回答。为每个问题标注期望记忆，记录召回是否相关，并分析关键词记忆服务在同义词、否定句和信息更新上的局限。

### 4. Artifact 版本化报告（工程）

让 Agent 生成一份报告并保存为 Artifact，随后根据用户反馈生成第二版；提供列出版本和读取指定版本的能力。报告正文不得塞进 State，只在 State 中保存文件名、版本号与摘要。验收效果是同一 Session 能准确引用两个版本，且大内容与小状态职责清晰。

### 5. 设计一份记忆治理策略（开放）

针对企业助理，规定哪些信息允许进入 Session、State、Memory 和 Artifact，补充保留期限、用户删除权、敏感信息脱敏、租户隔离与审计要求。再将方案映射到 LangGraph 的 Checkpointer 与 Store，指出两边最容易产生错误类比的地方。
